In [1]:
'''Gemma 4 | Budget-Aware Coding Agent - Test-Guided Repair

This notebook builds a single-agent submission.zip. The prior Budget-Aware
notebook used an analyzer and a coder; this version develops our 0.10
single-agent submission with a bounded test-feedback repair loop.
This candidate has not been scored. Model, sampling, declared tools and
evaluation limits are unchanged from Budget-Aware v2.

Public baseline supplied by the user: gemma4-agent-baseline-v2.ipynb.
Its original Kaggle author and URL are not recorded in the supplied copy;
add both before publishing this version. The source reported 0.13, while
our adaptation scored 0.10 on the public leaderboard.

Earlier Budget-Aware notebook adapted Roman Rozen's public baseline:
https://www.kaggle.com/code/romanrozen/gemma-eda-baseline-for-a-start-lb-top-1
That two-agent design is not used in this submission payload.

Run the cells in order to create submission.zip. This notebook only packages
the agent; it does not run competition evaluation or submit to Kaggle.
'''

from pathlib import Path
import hashlib
import os
import zipfile
import yaml

OUTPUT_DIR = Path(os.environ.get('GEMMA_OUTPUT_ROOT', '/kaggle/working'))
if not OUTPUT_DIR.is_dir() and 'GEMMA_OUTPUT_ROOT' not in os.environ:
    OUTPUT_DIR = Path.cwd() / 'gemma4_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = OUTPUT_DIR / 'submission.zip'


In [2]:
'''Add a bounded test-feedback loop to the scored single-agent baseline.'''

AGENT_YAML = 'name: swe_agent\nagent_class: LlmAgent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Software engineering agent that locates the cause of an issue, edits the source code, and submits a patch.\ninstruction: !include prompts/system.md\ngenerate_content_config: !include configs/sampling.yaml\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - search_similar_code\n  - get_code_neighbors\n  - get_code_subgraph\n'
SAMPLING_YAML = 'temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 8192\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n'
SYSTEM_PROMPT = '''\
You are an autonomous software engineer fixing one issue in a Python repository located at /workspace.
After you finish, hidden tests will be applied to your patch. The task counts as solved only if those tests pass.

## Budget
- The user message states your time allowance and tool-call allowance. When the time runs out, the session is stopped immediately.
- Edits saved to disk are kept as your patch even if the session is stopped. Therefore make your first concrete source edit before about half of the time allowance has passed, then verify and refine it.
- Call get_status at the start, after the first source edit, and before final submission. Use the reported remaining time and calls to decide how much checking is possible.
- If less than 60 seconds or five tool calls remain, stop exploring: keep the smallest defensible source patch, perform one bounded check if possible, then submit.

## Workflow
1. Read the problem statement and hints. Note every exact name they mention: functions, classes, parameters, default values, exception types, error messages, CLI options, file paths, and expected output.
2. Locate the relevant code with run_command and grep. Restrict the search to Python files and limit the output, for example:
   grep -rn "symbol_name" --include=*.py . | grep -v "/tests/" | head -n 30
   Use search_similar_code only when grep returns nothing useful.
3. Read only the relevant region with read_file, using start_line and end_line. Read a relevant caller when needed to understand the public contract. Do not read entire large files.
4. Look for the nearest existing tests for the affected behaviour and read only relevant cases. Use their calling conventions, assertions and fixtures to guide the patch. If a quick targeted test or reproduction can demonstrate the original failure, run it once before editing; skip this preliminary run if setup would delay the first source edit. Do not run the whole test suite.
5. Edit with edit_file. Copy old_string exactly from the read_file output, including indentation, and include 2 to 4 surrounding lines so that the match is unique.
6. Run one focused existing test or a minimal reproduction in /tmp after the patch. For example: timeout 30s python3 -m pytest tests/test_module.py -x -q -k keyword --tb=short. Use an explicit command timeout no longer than 30 seconds and shorter if needed to reserve at least 30 seconds for finalization. Keep enough output to see the failing assertion and relevant traceback.
   If the check fails because of the requested behaviour or a regression introduced by your patch, use that specific failure to revise the source and rerun the same check. Allow at most two repair-and-recheck cycles after the initial check. Before each cycle, call get_status and proceed only if at least 60 seconds and six tool calls remain. Fix the cause; do not weaken assertions or modify tests.
   Stop when the focused check passes. Missing dependencies, collection errors and timeouts are inconclusive, not successful checks: do not install packages or repeatedly rerun an unchanged command. Ignore an unrelated failure only when there is evidence that it already existed before your edit. If no runnable check is available, inspect the changed logic and relevant callers, then finalize without claiming tests passed.
7. Inspect git diff and git status --short to confirm the intended source patch exists. Call submit_patch as the final intended tool action. The harness may end immediately after that call, so do not rely on a later confirmation or another edit.

## Rules that decide whether the hidden tests pass
- Use names, signatures, default values, exception types, and message strings exactly as written in the issue. The hidden tests import and compare them literally.
- For a feature request, implement the complete public behaviour: new parameters, return values, exports in __init__.py, and the type hints used in the project.
- Keep backward compatibility. Do not rename or remove existing public functions, classes, or parameters.
- For FastAPI tasks that mention documentation examples, the executable examples are under docs_src/ and are imported by the tests. Create or update them when the issue requires it.
- Never create, modify, or delete test files, conftest.py, or pytest.ini. Change packaging files such as pyproject.toml only when the issue explicitly requires it.
- Write scratch scripts only under /tmp. Every new file under /workspace becomes part of your patch.
- Do not run pip install. The environment is offline and all dependencies are already installed.
- Do not run bare pytest or the full test suite. It exceeds the command timeout.

## Style
Keep your reasoning short. Call a tool in every turn until you have submitted the patch.
'''
EVAL_YAML = '''evaluation:
  timeout_seconds: 90
  max_tool_calls: 60
  max_time_minutes: 4
  max_turns: 80
'''
FILES = {
    'agent.yaml': AGENT_YAML,
    'configs/sampling.yaml': SAMPLING_YAML,
    'eval_config.yaml': EVAL_YAML,
    'prompts/system.md': SYSTEM_PROMPT,
}


In [3]:
'''Validate includes, declared tools, and the 12-hour planning scenario.'''

class IncludeLoader(yaml.SafeLoader):
    pass

def include(loader, node):
    relative = loader.construct_scalar(node)
    assert relative in FILES, relative
    return FILES[relative] if relative.endswith('.md') else yaml.safe_load(FILES[relative])

IncludeLoader.add_constructor('!include', include)
agent = yaml.load(AGENT_YAML, Loader=IncludeLoader)
evaluation = yaml.safe_load(EVAL_YAML)['evaluation']
assert agent['model'] == 'gemma-4-31b-it-qat-w4a16-ct'
assert agent['instruction'] == SYSTEM_PROMPT
assert agent['generate_content_config'] == yaml.safe_load(SAMPLING_YAML)
assert set(agent['tools']) == {
    'run_command', 'read_file', 'edit_file', 'write_file', 'get_status',
    'submit_patch', 'search_similar_code', 'get_code_neighbors', 'get_code_subgraph',
}
assert evaluation == {
    'timeout_seconds': 90,
    'max_tool_calls': 60,
    'max_time_minutes': 4,
    'max_turns': 80,
}
assert '{' not in SYSTEM_PROMPT and '}' not in SYSTEM_PROMPT
scenario_minutes = 130 * (evaluation['max_time_minutes'] + 0.5) + 20
print(f'Planning scenario: {scenario_minutes:.0f} minutes of 720; setup and task count are estimates.')
print('Configuration checks passed.')


Planning scenario: 605 minutes of 720; setup and task count are estimates.
Configuration checks passed.


In [4]:
'''Package the four submission files at the ZIP root.'''

with zipfile.ZipFile(ZIP_PATH, 'w') as archive:
    for relative, content in sorted(FILES.items()):
        entry = zipfile.ZipInfo(relative, date_time=(1980, 1, 1, 0, 0, 0))
        entry.compress_type = zipfile.ZIP_DEFLATED
        entry.external_attr = 0o100644 << 16
        archive.writestr(entry, content.encode('utf-8'))
with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(FILES)
    assert archive.read('prompts/system.md') == SYSTEM_PROMPT.encode('utf-8')
print('Submission:', ZIP_PATH)
print('SHA-256:', hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
print('Archive files:', sorted(FILES))
print('Official model compilation and leaderboard scoring were not run.')


Submission: /Users/flexonafft/MLKaggleTasks/competitions/public-comp/gemma-4-developer-agent/artifacts/test_guided_repair/submission.zip
SHA-256: e979e612a13619d2489c76c54dd678c8596d2672989bc46ca5c5fd75804937b9
Archive files: ['agent.yaml', 'configs/sampling.yaml', 'eval_config.yaml', 'prompts/system.md']
Official model compilation and leaderboard scoring were not run.
